In [1]:
# this is the copy version of the one in the FMNIST

Hello


In [2]:
''' This notebook takes the all the runs and create a file with average of CE_test, CE_train, and accuracy (test) and creates a file called
 averaged_runs_p_0.0_bs_1024 foe all pruning percentage and all the batch size inside the directories. 

 Then based on the file, it will create 6 plots with will combined Avg_CE_test/ Avg_CE_train  vs Batch number for each batch size ( 64, 1024, 60000 ). 

 The plot is named as : CE_Train_Avg_SLP_FMNIST_BS_1024

'''

' This notebook takes the all the runs and create a file with average of CE_test, CE_train, and accuracy (test) and creates a file called\n averaged_runs_p_0.0_bs_1024 foe all pruning percentage and all the batch size inside the directories. \n\n Then based on the file, it will create 6 plots with will combined Avg_CE_test/ Avg_CE_train  vs Batch number for each batch size ( 64, 1024, 60000 ). \n\n The plot is named as : CE_Train_Avg_SLP_MNIST_BS_1024\n\n'

In [ ]:
"""
Loads averaged training results for Convolutional FMNIST model
under different pruning-layer configurations and batch sizes,
and generates publication-quality plots.

For each pruning configuration (CONV, FHL, SHL, FHL+SHL, ALL)
and each batch size (64, 1024), the script:

1. Loads averaged CSV files for pruning levels (0%-100%).
2. Extracts Avg CE Train, Avg CE Test, Avg Accuracy.
3. Generates three plots per configuration.
4. Saves all figures at 300 DPI in Avg_Plots/{prune_layer}/.

Run Untitled.ipynb first to generate the averaged CSV files.
"""

import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
BASE_DIR_ROOT = r"C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-FMNIST"

PRUNE_LAYERS_OPTIONS = ['CONV', 'FHL', 'SHL', 'FHL+SHL', 'ALL']

PRUNE_LAYER_DIR_MAP = {
    "CONV":    "prune_layers_CONV",
    "FHL":     "prune_layers_FHL",
    "SHL":     "prune_layers_SHL",
    "FHL+SHL": "prune_layers_FHL+SHL",
    "ALL":     "prune_layers_ALL"
}

BATCH_DIR_TEMPLATE = "p-percentage_{:.1f}\\batch_size_{}"
AVG_FILE_PATTERN = "averaged_runs_conv_*_p_{}_bs_{}.csv"

BATCH_SIZES = [64, 1024]
PRUNING_LEVELS = [round(x * 0.1, 1) for x in range(11)]

LN10 = np.log(10)

# =========================
# COLOR LIST
# =========================
COLOR_LIST = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#800080",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#B9D9EB", "#17becf"
][::-1]

# =========================
# OUTPUT DIRECTORY
# =========================
PLOT_ROOT = os.path.join(BASE_DIR_ROOT, "Avg_Plots")
os.makedirs(PLOT_ROOT, exist_ok=True)

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 14
})

# =========================
# MAIN LOOP
# =========================
for prune_layer in PRUNE_LAYERS_OPTIONS:

    base_dir = os.path.join(BASE_DIR_ROOT, PRUNE_LAYER_DIR_MAP[prune_layer])
    plot_dir = os.path.join(PLOT_ROOT, prune_layer)
    os.makedirs(plot_dir, exist_ok=True)

    for bs in BATCH_SIZES:

        avg_dfs = {}

        # -------------------------
        # LOAD DATA
        # -------------------------
        for p in PRUNING_LEVELS:
            folder = os.path.join(base_dir, BATCH_DIR_TEMPLATE.format(p, bs))
            files = glob.glob(os.path.join(folder, AVG_FILE_PATTERN.format(p, bs)))

            if files:
                avg_dfs[p] = pd.read_csv(files[0])

        if not avg_dfs:
            continue

        sorted_items = sorted(avg_dfs.items())

        # ==========================================================
        # FUNCTION TO CREATE EACH PLOT
        # ==========================================================
        def create_plot(y_column, ylabel, ylim, subtitle, filename):

            fig, ax = plt.subplots(figsize=(12, 6))

            for idx, (p, df) in enumerate(sorted_items):
                color = COLOR_LIST[idx % len(COLOR_LIST)]
                ax.plot(
                    df["Batch_Number"],
                    df[y_column],
                    label=f"P%={int(p * 100)}",
                    color=color
                )

            ax.set_xlabel("Batch Number")
            ax.set_ylabel(ylabel)
            ax.set_ylim(*ylim)

            ax.text(
                0.5, 0.97,
                f"Convolutional FMNIST ({prune_layer}-layer(s))",
                transform=ax.transAxes,
                ha="center", va="top"
            )
            ax.text(
                0.5, 0.92,
                f"{subtitle}, BS={bs}",
                transform=ax.transAxes,
                ha="center", va="top",
                fontsize=16
            )

            if "CE" in y_column:
                ax.text(
                    0.06, LN10, r"$\ln(10)$",
                    transform=ax.get_yaxis_transform(),
                    fontsize=14, va="center"
                )

            handles, labels = ax.get_legend_handles_labels()
            ax.legend(handles[::-1], labels[::-1], frameon=False, loc="upper right")

            ax.grid(True)
            plt.tight_layout()

            fig.savefig(os.path.join(plot_dir, filename), dpi=300, bbox_inches="tight")
            plt.show()
            plt.close(fig)

        # -------------------------
        # CE TRAIN
        # -------------------------
        create_plot(
            y_column="Avg_CE_Train",
            ylabel="Average CE",
            ylim=(0, 2.7),
            subtitle="Training-Vectors",
            filename=f"Conv_Avg_CE_Train_{prune_layer}_BS{bs}.png"
        )

        # -------------------------
        # CE TEST
        # -------------------------
        create_plot(
            y_column="Avg_CE_Test",
            ylabel="Average CE",
            ylim=(0, 2.7),
            subtitle="Test-Vectors",
            filename=f"Conv_Avg_CE_Test_{prune_layer}_BS{bs}.png"
        )

        # -------------------------
        # ACCURACY
        # -------------------------
        create_plot(
            y_column="Avg_Accuracy",
            ylabel="Average Accuracy (%)",
            ylim=(0, 100),
            subtitle="Test-Vectors",
            filename=f"Conv_Avg_Accuracy_{prune_layer}_BS{bs}.png"
        )

        print(f"[DONE] {prune_layer}, BS={bs}")